In [18]:
from pathlib import Path
import pandas as pd

In [19]:

# 1. Xác định thư mục data
ROOT = Path.cwd()
DATA_DIR = ROOT/ ".." / "data" / "raw"

# 2. Đọc dữ liệu CSV
CSV_PATH = DATA_DIR / "car_dataset.csv"

df = pd.read_csv(CSV_PATH)
print(df.head())

                       name  year  selling_price  km_driven    fuel  \
0             Maruti 800 AC  2007          60000      70000  Petrol   
1  Maruti Wagon R LXI Minor  2007         135000      50000  Petrol   
2      Hyundai Verna 1.6 SX  2012         600000     100000  Diesel   
3    Datsun RediGO T Option  2017         250000      46000  Petrol   
4     Honda Amaze VX i-DTEC  2014         450000     141000  Diesel   

  seller_type transmission         owner  
0  Individual       Manual   First Owner  
1  Individual       Manual   First Owner  
2  Individual       Manual   First Owner  
3  Individual       Manual   First Owner  
4  Individual       Manual  Second Owner  


In [20]:
# xóa dòng trùng lặp
df = df.drop_duplicates().reset_index(drop=True)
print(f"After deduplication: {df.shape[0]} rows")

After deduplication: 3577 rows


In [21]:
# 3.5  Feature Engineering 

# vehicle_age is more informative than raw year for regression
REFERENCE_YEAR = 2026  # dataset collected up to 2026
df['vehicle_age'] = REFERENCE_YEAR - df['year']
# Trích xuất tên hãng
df['brand'] = df['name'].str.split().str[0]

# Chỉ xử lý Land Rover (Từ đặc biệt)
df.loc[
    df['name'].str.startswith('Land Rover'),
    'brand'
] = 'Land Rover'

# drop raw columns replaced by engineered ones
df.drop(columns=['name', 'year'], inplace=True)

print("Engineered features added: vehicle_age, brand")
df.head(3)

Engineered features added: vehicle_age, brand


C:\Users\TNC\AppData\Local\Temp\ipykernel_6412\3028600396.py:16: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['name', 'year'], inplace=True)


,selling_price,km_driven,fuel,seller_type,transmission,owner,vehicle_age,brand
0,60000,70000,Petrol,Individual,Manual,First Owner,19,Maruti
1,135000,50000,Petrol,Individual,Manual,First Owner,19,Maruti
2,600000,100000,Diesel,Individual,Manual,First Owner,14,Hyundai


In [22]:
# 1. Xác định thư mục lưu dữ liệu
ROOT = Path.cwd()
DATA_DIR = ROOT / ".." / "data" / "processed"
# 3. Xác định đường dẫn file CSV
CSV_PATH = DATA_DIR / "car_cleaned.csv"

# 4. Lưu dữ liệu sau preprocessing
df.to_csv(CSV_PATH, index=False)

Feature Engineering – Brand Extraction

Cột name chứa thông tin về hãng, dòng xe và phiên bản xe, với 1.491 tên xe khác nhau.

Tiến hành trích xuất đặc trưng brand từ phần đầu của tên xe và xử lý trường hợp đặc biệt Land Rover.

Kết quả: Tạo được 29 nhóm tên hãng hoặc tên xe được trích xuất từ cột name.

Mục đích: Giảm số lượng giá trị phân loại cần mã hóa, đồng thời giữ lại thông tin về hãng xe để phục vụ dự đoán giá bán.

Đặc trưng brand sẽ được mã hóa trong giai đoạn Data Preprocessing trước khi huấn luyện mô hình.